# Chinook Music Store Exploration  
**Dataset:** [Chinook Dataset (Github)](https://github.com/lerocha/chinook-database)   
*Exploratory Data Analysis*

## Project Summary

This project explores the Chinook digital music store database (11 relational tables: artists, albums, tracks, genres, media types, employees, customers, invoices, and invoice lines) in PostgreSQL, queried through Jupyter using `%%sql` magic.

It moves from schema discovery, through core filtering and aggregation, into joins, subqueries, and window functions, and closes with business questions on sales performance and customer spend, including which sales agent generates the most revenue, which customers spend above average, and where any anomalies exist in staff reporting lines.

Full write-up (approach, SQL concepts used, how missing and duplicate data were handled, KPI recommendation, and challenges faced) is in the project README.


In [25]:
## re
import os
from dotenv import load_dotenv

load_dotenv()
%load_ext sql
%sql $DATABASE_URL

%config SqlMagic.feedback = False
%config SqlMagic.displaylimit = None

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


**All tables in the database**

In [27]:
%%sql

SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;

table_name
album
artist
customer
employee
genre
invoice
invoice_line
media_type
playlist
playlist_track


**EMPLOYEES WHO HOLD THE JOB TITLE 'SALES SUPPORT AGENT'**

In [28]:
%%sql
SELECT 
  first_name, last_name,title, hire_date
FROM employee
WHERE title = 'Sales Support Agent'


first_name,last_name,title,hire_date
Jane,Peacock,Sales Support Agent,2002-04-01 00:00:00
Margaret,Park,Sales Support Agent,2003-05-03 00:00:00
Steve,Johnson,Sales Support Agent,2003-10-17 00:00:00


**TRACKS LONGER THAN 5 MINUTES(300,000 MILLISECONDS) SORTING FROM THE LONGEST TO THE SHORTEST**

In [29]:
%%sql
SELECT name, minutes
FROM track
WHERE minutes > 5
ORDER BY minutes DESC
LIMIT 10;

name,minutes
Occupation / Precipice,88.12
Through a Looking Glass,84.81
"Greetings from Earth, Pt. 1",49.34
The Man With Nine Lives,49.28
"Battlestar Galactica, Pt. 2",49.27
"Battlestar Galactica, Pt. 1",49.21
Murder On the Rising Star,48.93
"Battlestar Galactica, Pt. 3",48.80
Take the Celestra,48.79
Fire In Space,48.78


**CUSTOMERS WHO LIVE IN BRAZIL, CANADA, GERMANY**

In [30]:
%%sql
SELECT first_name, last_name
FROM customer
WHERE country IN ('Brazil', 'Canada', 'Germany')
LIMIT 10;

first_name,last_name
Luís,Gonçalves
Leonie,Köhler
François,Tremblay
Eduardo,Martins
Alexandre,Rocha
Roberto,Almeida
Fernanda,Ramos
Mark,Philips
Jennifer,Peterson
Robert,Brown


**HOW MANY CUSTOMERS DOES THE STORE HAVE IN EACH COUNTRY**

In [31]:
%%sql
SELECT country, COUNT(*) AS total_customers
FROM customer
GROUP BY country
ORDER BY country DESC
LIMIT 10;

country,total_customers
USA,13
United Kingdom,3
Sweden,1
Spain,1
Portugal,2
Poland,1
Norway,1
Netherlands,1
Italy,1
Ireland,1


 **INVOICES HAD A TOTAL SPENDING GREATER THAN 10.00**

In [32]:
%%sql
SELECT invoice_id, total
FROM invoice
WHERE total > 10.00
LIMIT 10;

invoice_id,total
5,13.86
12,13.86
19,13.86
26,13.86
33,13.86
40,13.86
47,13.86
54,13.86
61,13.86
68,13.86


**LIST OF TRACK NAMES ALONGSIDE THE ALBUM THEY BELONG TO**

In [33]:
%%sql
SELECT t.name AS track_names, a.title AS album_title
FROM track AS t 
INNER JOIN album AS a 
ON t.album_id = a.album_id
LIMIT 10;

track_names,album_title
Go Down,Let There Be Rock
Blind Man,Big Ones
Deuces Are Wild,Big Ones
The Other Side,Big Ones
Crazy,Big Ones
Eat The Rich,Big Ones
Angel,Big Ones
Livin' On The Edge,Big Ones
All I Really Want,Jagged Little Pill
You Oughta Know,Jagged Little Pill


**WHICH MEDIA TYPE IS THE MOST POPULAR(HOW MANY TRACKS USE EACH)**

In [34]:
%%sql
SELECT m.name AS media_type, COUNT(t.track_id) AS track_count
FROM media_type AS m
LEFT JOIN track AS t
ON m.media_type_id = t.media_type_id
GROUP BY m.name
ORDER BY track_count DESC;

media_type,track_count
MPEG audio file,3034
Protected AAC audio file,237
Protected MPEG-4 video file,214
AAC audio file,11
Purchased AAC audio file,7


**REPORT SHOWING CUSTOMER NAMES, AND THE TOTAL AMOUNT THEY HAVE SPENT ACROSS ALL THEIR INVOICES**

In [35]:
%%sql
SELECT c.first_name || ' ' || c.last_name AS customer_name, i.total AS total_amount_spent
FROM customer AS c
INNER JOIN invoice AS i
ON c.customer_id = i.customer_id
LIMIT 10;

customer_name,total_amount_spent
Leonie Köhler,1.98
Bjørn Hansen,3.96
Daan Peeters,5.94
Mark Philips,8.91
John Gordon,13.86
Fynn Zimmermann,0.99
Niklas Schröder,1.98
Dominique Lefebvre,1.98
Wyatt Girard,3.96
Hugh O'Reilly,5.94


**TRACKS THAT BELONG TO THE DRAMA GENRE**

In [36]:
%%sql
SELECT t.track_id, t.name AS track_name, g.name AS genre
FROM track AS t
INNER JOIN genre AS g 
ON t.genre_id = g.genre_id
AND g.name = 'Drama'
LIMIT 10;

track_id,track_name,genre
2840,Don't Look Back,Drama
2841,One Giant Leap,Drama
2842,Collision,Drama
2843,Hiros,Drama
2844,Better Halves,Drama
2846,Seven Minutes to Midnight,Drama
2847,Homecoming,Drama
2849,Fallout,Drama
2850,The Fix,Drama
2851,Distractions,Drama


**USING A SUBQUERY; DRAMA GENRE**

In [37]:
%%sql
SELECT t.track_id, t.name AS name
FROM track AS t
WHERE t.genre_id = (SELECT g.genre_id
                    FROM genre AS g 
                    WHERE g.name = 'Drama')
LIMIT 10;

track_id,name
2840,Don't Look Back
2841,One Giant Leap
2842,Collision
2843,Hiros
2844,Better Halves
2846,Seven Minutes to Midnight
2847,Homecoming
2849,Fallout
2850,The Fix
2851,Distractions


**TOTAL SALES MADE BY SALES AGENT (EMPLOYEE) ORDER DESC**

In [38]:
%%sql
SELECT e.first_name || ' ' || e.last_name AS employee_name,
    SUM(i.total) AS total_sales
FROM employee AS e 
INNER JOIN customer AS e_cust
    ON e.employee_id = e_cust.support_rep_id
INNER JOIN invoice AS i
    ON e_cust.customer_id = i.customer_id
GROUP BY employee_name
ORDER BY total_sales DESC;

employee_name,total_sales
Jane Peacock,833.04
Margaret Park,775.40
Steve Johnson,720.16


**WHICH EMPLOYEES HAVE A HIGHER RANK THAN THEIR MANAGER?**   

*Derive rank explicitly from job title (General Manager outranks any other Manager title, which outranks all Staff/Agent titles), then compare each employee's rank number to their manager's rank number. A LOWER rank number means a HIGHER rank, so an employee outranks their manager only when their rank number is less than their manager's.*

In [39]:
%%sql
WITH ranked_employees AS (
   SELECT employee_id,
          first_name || ' ' || last_name AS employee_name,
          title,
          reports_to,
    CASE
        WHEN title = 'General Manager' THEN 1
        WHEN title LIKE '%Manager%' THEN 2
        ELSE 3
     END AS rank_level
    FROM employee
)
SELECT
    e.employee_name,
    e.title AS employee_title,
    e.rank_level AS employee_rank_level,
    m.employee_name AS manager_name,
    m.title AS manager_title,
    m.rank_level AS manager_rank_level
FROM ranked_employees AS e
INNER JOIN ranked_employees AS m ON e.reports_to = m.employee_id
WHERE e.rank_level < m.rank_level
LIMIT 10;

employee_name,employee_title,employee_rank_level,manager_name,manager_title,manager_rank_level


Note: employee.reports_to is NULL for exactly one employee, the General Manager, who has no manager to report to. Since this query joins each employee to their manager, that employee is automatically excluded from the result (an INNER JOIN can't match a NULL). 

**CUSTOMERS WHO HAVE SPENT  MORE THAN THE AVERAGE SPENDING OF ALL CUSTOMERS**

In [40]:
%%sql
SELECT i.invoice_id, 
       i.customer_id, 
       i.total AS customer_spent 
FROM invoice AS i
WHERE i.total >(SELECT AVG(total)
                FROM invoice)
LIMIT 10;

invoice_id,customer_id,customer_spent
3,8,5.94
4,14,8.91
5,23,13.86
10,46,5.94
11,52,8.91
12,2,13.86
17,25,5.94
18,31,8.91
19,40,13.86
24,4,5.94


**CUSTOMERS WHO HAVE SPENT  MORE THAN THE AVERAGE SPENDING OF ALL CUSTOMERS**

In [41]:
%%sql
SELECT c.first_name || ' ' || c.last_name AS customer_name,
       i.invoice_id, 
       i.total AS customer_spent 
FROM customer AS c 
INNER JOIN invoice AS i 
ON c.customer_id = i.customer_id
WHERE i.total > ( SELECT AVG(total)
                 FROM invoice
                )
ORDER BY i.total DESC
LIMIT 10;

customer_name,invoice_id,customer_spent
Helena Holý,404,25.86
Richard Cunningham,299,23.86
Ladislav Kovács,96,21.86
Hugh O'Reilly,194,21.86
Victor Stevens,201,18.86
Astrid Gruber,89,18.86
Luis Rojas,88,17.91
František Wichterlová,306,16.86
Isabelle Mercier,313,16.86
Bjørn Hansen,208,15.86


**EVERY TRACK NAME, ITS ALBUM, ITS LENGTH, AND MAXIMUM TRACK LENGTH FOUND IN THAT SPECIFIC ALBUM**

In [42]:
%%sql
SELECT name AS track_name, 
       album_id, minutes AS track_length, 
       MAX(minutes) OVER(PARTITION BY album_id) AS max_track_in_album
FROM track
LIMIT 10;

track_name,album_id,track_length,max_track_in_album
C.O.D.,1,3.33,5.73
Breaking The Rules,1,4.39,5.73
Night Of The Long Knives,1,3.43,5.73
Spellbound,1,4.51,5.73
For Those About To Rock (We Salute You),1,5.73,5.73
Put The Finger On You,1,3.43,5.73
Let's Get It Up,1,3.90,5.73
Inject The Venom,1,3.51,5.73
Snowballed,1,3.39,5.73
Evil Walks,1,4.39,5.73


**RANK TRACKS WITHIN EACH GENRE BY THEIR PRICE OR LENGTH RESETTING THE RANK BACK TO 1 WHEN A NEW GENRE STARTS**

In [43]:
%%sql
SELECT t.genre_id, 
       t.name AS track_name, 
       t.minutes AS track_length,
       DENSE_RANK() OVER(
       PARTITION BY t.genre_id
       ORDER BY t.minutes DESC
       ) AS track_rank_in_genre
FROM track AS t
LIMIT 10;

genre_id,track_name,track_length,track_rank_in_genre
1,Dazed And Confused,26.87,1
1,Space Truckin',19.93,2
1,Dazed And Confused,18.61,3
1,We've Got To Get Together/Jingo,17.83,4
1,Funky Piano,15.58,5
1,Going Down / Highway Star,15.23,6
1,Santana Jam,14.71,7
1,The Sun Road,14.68,8
1,Whole Lotta Love,14.40,9
1,Mistreated (Alternate Version),14.25,10
